# 1. Import Library dan Setup
Bagian ini digunakan untuk mengimpor semua library yang dibutuhkan untuk proses pemodelan, seperti `scikit-learn` untuk preprocessing/split, `xgboost`, dan `tensorflow.keras` untuk deep learning.

In [1]:
import numpy as np
import scipy.sparse as sp
import joblib
import os
import tensorflow as tf
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, GlobalMaxPooling1D, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

# 2. Load Extracted Features dan Labels
Membaca fitur-fitur yang telah diekstrak pada tahapan sebelumnya (TF-IDF, Word2Vec, GloVe, dan FastText) beserta target labels dari folder `extracted_features/`.

In [2]:
# Load data
feature_dir = 'extracted_features'

# Memuat label
y = np.load(os.path.join(feature_dir, 'y_labels.npy'))

# Memuat semua fitur
X_tfidf = sp.load_npz(os.path.join(feature_dir, 'X_tfidf.npz')).todense() # Convert sparse to dense array
X_w2v = np.load(os.path.join(feature_dir, 'X_w2v.npy'))
X_glove = np.load(os.path.join(feature_dir, 'X_glove.npy'))
X_ft = np.load(os.path.join(feature_dir, 'X_ft.npy'))

print(f"Bentuk TF-IDF    : {X_tfidf.shape}")
print(f"Bentuk Word2Vec  : {X_w2v.shape}")
print(f"Bentuk GloVe     : {X_glove.shape}")
print(f"Bentuk FastText  : {X_ft.shape}")
print(f"Bentuk Label     : {y.shape}")

Bentuk TF-IDF    : (20000, 839)
Bentuk Word2Vec  : (20000, 100)
Bentuk GloVe     : (20000, 100)
Bentuk FastText  : (20000, 100)
Bentuk Label     : (20000,)


# 3. Split Data (Train & Test)
Membagi masing-masing set fitur menjadi data latih (80%) dan data uji (20%) menggunakan parameter `random_state` yang sama agar pembagian observasinya konsisten pada setiap eksperimen.

In [3]:
# Kita akan membuat dictionary untuk memudahkan iterasi
features_dict = {
    'TF-IDF': np.asarray(X_tfidf),
    'Word2Vec': X_w2v,
    'GloVe': X_glove,
    'FastText': X_ft
}

data_splits = {}

for name, feature_matrix in features_dict.items():
    X_train, X_test, y_train, y_test = train_test_split(feature_matrix, y, test_size=0.2, random_state=42)
    data_splits[name] = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }
    print(f"[{name}] Train: {X_train.shape}, Test: {X_test.shape}")

[TF-IDF] Train: (16000, 839), Test: (4000, 839)
[Word2Vec] Train: (16000, 100), Test: (4000, 100)
[GloVe] Train: (16000, 100), Test: (4000, 100)
[FastText] Train: (16000, 100), Test: (4000, 100)


# 4. Fungsi Evaluasi Model
Mendefinisikan fungsi pembantu untuk menyimpan log performa, agar nantinya dapat digunakan pada evaluasi akhir keseluruhan model.

In [4]:
results = []

def evaluate_model(model_name, feature_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    results.append({
        'Model': model_name,
        'Feature': feature_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    print(f"\n[{model_name} | {feature_name}]")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")

# 5. Pemodelan dengan XGBoost
Algoritma berbasis tree-ensemble ini cukup handal dalam klasifikasi. XGBoost akan dilatih dengan keempat varian representasi teks.

In [5]:
for feature_name in features_dict.keys():
    X_train = data_splits[feature_name]['X_train']
    X_test = data_splits[feature_name]['X_test']
    y_train = data_splits[feature_name]['y_train']
    y_test = data_splits[feature_name]['y_test']
    
    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb.fit(X_train, y_train)
    y_pred = xgb.predict(X_test)
    
    evaluate_model('XGBoost', feature_name, y_test, y_pred)


[XGBoost | TF-IDF]
Accuracy  : 0.5095
Precision : 0.5098
Recall    : 0.5095
F1-Score  : 0.5095

[XGBoost | Word2Vec]
Accuracy  : 0.4950
Precision : 0.4950
Recall    : 0.4950
F1-Score  : 0.4950

[XGBoost | GloVe]
Accuracy  : 0.5005
Precision : 0.5005
Recall    : 0.5005
F1-Score  : 0.5005

[XGBoost | FastText]
Accuracy  : 0.5118
Precision : 0.5117
Recall    : 0.5118
F1-Score  : 0.5117


# 6. Pemodelan dengan CNN (Convolutional Neural Network)
CNN dapat menangkap relasi spasial dalam sequence data. Untuk Deep Learning (CNN dan LSTM) yang mensyaratkan input sequence 3D `(batch_size, sequence_length, embedding_dim)`, kita akan mereshape matriks fitur teks (berdimensi `N x Dimensi_Fitur`) menjadi sekumpulan sequence.

In [6]:
def create_cnn_model(input_shape):
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, padding='same', activation='relu', input_shape=input_shape),
        GlobalMaxPooling1D(),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

for feature_name in features_dict.keys():
    X_train = data_splits[feature_name]['X_train']
    X_test = data_splits[feature_name]['X_test']
    y_train = data_splits[feature_name]['y_train']
    y_test = data_splits[feature_name]['y_test']
    
    # Reshape (samples, features, 1) agar bisa masuk ke layer Conv1D
    X_train_cnn = np.expand_dims(X_train, axis=2)
    X_test_cnn = np.expand_dims(X_test, axis=2)
    
    model = create_cnn_model((X_train_cnn.shape[1], 1))
    
    # Gunakan verbosity 0 untuk mencegah output yang terlalu panjang
    model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, 
              validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_probs = model.predict(X_test_cnn, verbose=0)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()
    
    evaluate_model('CNN', feature_name, y_test, y_pred)


[CNN | TF-IDF]
Accuracy  : 0.5072
Precision : 0.2573
Recall    : 0.5072
F1-Score  : 0.3414

[CNN | Word2Vec]
Accuracy  : 0.4928
Precision : 0.2428
Recall    : 0.4928
F1-Score  : 0.3253

[CNN | GloVe]
Accuracy  : 0.4928
Precision : 0.2428
Recall    : 0.4928
F1-Score  : 0.3253

[CNN | FastText]
Accuracy  : 0.5072
Precision : 0.2573
Recall    : 0.5072
F1-Score  : 0.3414


# 7. Pemodelan dengan LSTM (Long Short-Term Memory)
LSTM sangat tangguh dalam menangkap dependensi jangka panjang pada sequence data. Kita akan memperlakukan fitur seperti sequence time-step agar cocok dengan layer LSTM.

In [7]:
def create_lstm_model(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=False, input_shape=input_shape),
        Dropout(0.5),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

for feature_name in features_dict.keys():
    X_train = data_splits[feature_name]['X_train']
    X_test = data_splits[feature_name]['X_test']
    y_train = data_splits[feature_name]['y_train']
    y_test = data_splits[feature_name]['y_test']
    
    # Reshape (samples, 1, features)
    X_train_lstm = np.expand_dims(X_train, axis=1)
    X_test_lstm = np.expand_dims(X_test, axis=1)
    
    model = create_lstm_model((1, X_train.shape[1]))
    
    model.fit(X_train_lstm, y_train, epochs=10, batch_size=32, 
              validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_probs = model.predict(X_test_lstm, verbose=0)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()
    
    evaluate_model('LSTM', feature_name, y_test, y_pred)


[LSTM | TF-IDF]
Accuracy  : 0.5095
Precision : 0.5090
Recall    : 0.5095
F1-Score  : 0.4343

[LSTM | Word2Vec]
Accuracy  : 0.4928
Precision : 0.2428
Recall    : 0.4928
F1-Score  : 0.3253

[LSTM | GloVe]
Accuracy  : 0.4885
Precision : 0.4832
Recall    : 0.4885
F1-Score  : 0.3869

[LSTM | FastText]
Accuracy  : 0.4928
Precision : 0.2428
Recall    : 0.4928
F1-Score  : 0.3253


In [8]:
# simpan semua model LSTM, XGBoost, dan CNN dengan format .pkl

# 8. Evaluasi dan Perbandingan Keseluruhan
Menyusun semua hasil eksperimen ke dalam Pandas DataFrame untuk membandingkan secara eksplisit performa masing-masing model yang dikombinasikan dengan jenis ekstraksi fitur yang berbeda.

In [9]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['F1-Score', 'Accuracy'], ascending=False).reset_index(drop=True)

print("=== PERBANDINGAN MODEL DAN FEATURE EXTRACTION ===")
display(results_df)

=== PERBANDINGAN MODEL DAN FEATURE EXTRACTION ===


,Model,Feature,Accuracy,Precision,Recall,F1-Score
0,XGBoost,FastText,0.51175,0.511711,0.51175,0.511727
1,XGBoost,TF-IDF,0.50950,0.509768,0.50950,0.509469
2,XGBoost,GloVe,0.50050,0.500500,0.50050,0.500500
3,XGBoost,Word2Vec,0.49500,0.495029,0.49500,0.495013
4,LSTM,TF-IDF,0.50950,0.508987,0.50950,0.434271
5,LSTM,GloVe,0.48850,0.483166,0.48850,0.386946
6,CNN,TF-IDF,0.50725,0.257303,0.50725,0.341420
7,CNN,FastText,0.50725,0.257303,0.50725,0.341420
8,CNN,Word2Vec,0.49275,0.242803,0.49275,0.325309
9,CNN,GloVe,0.49275,0.242803,0.49275,0.325309


# 9. Menyimpan Model Terbaik
Dari hasil perbandingan di atas, model beserta jenis representasi vektornya yang memberikan F1-Score atau Accuracy terbaik akan disimpan di dalam direktori `Model/` untuk dapat digunakan pada saat deployment (Gradio / Streamlit).

In [10]:
# Pastikan folder model ada
os.makedirs('../Model', exist_ok=True)

best_model_idx = results_df.index[0]
best_model_name = results_df.loc[best_model_idx, 'Model']
best_feature_name = results_df.loc[best_model_idx, 'Feature']

print(f"Model terbaik yang akan disimpan adalah: {best_model_name} dengan fitur {best_feature_name}")

# PENTING:
# Di dunia nyata, Anda harus melatih ulang model tersebut pada KESELURUHAN DATA
# atau menyimpan instansi model keras/sklearn yang mendapatkan hasil terbaik tadi.
# Snippet di bawah ini adalah contoh re-train dan save menggunakan XGBoost.

# Jika yang terbaik adalah XGBoost:
if best_model_name == 'XGBoost':
    print("Menyimpan model XGBoost...")
    best_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    best_xgb.fit(data_splits[best_feature_name]['X_train'], data_splits[best_feature_name]['y_train'])
    joblib.dump(best_xgb, '../Model/best_fake_news_model.joblib')
    print("Model disimpan ke '../Model/best_fake_news_model.joblib'")

# Jika Deep Learning, gunakan model.save()
elif best_model_name in ['CNN', 'LSTM']:
    print(f"Menyimpan model {best_model_name}...")
    # Latih model final (Contoh singkat, sebaiknya gunakan instance terbaik yang dihasilkan loop atas)
    if best_model_name == 'CNN':
        model_final = create_cnn_model((data_splits[best_feature_name]['X_train'].shape[1], 1))
        X_train_final = np.expand_dims(data_splits[best_feature_name]['X_train'], axis=2)
    else:
        model_final = create_lstm_model((1, data_splits[best_feature_name]['X_train'].shape[1]))
        X_train_final = np.expand_dims(data_splits[best_feature_name]['X_train'], axis=1)
        
    model_final.fit(X_train_final, data_splits[best_feature_name]['y_train'], epochs=5, batch_size=32, verbose=0)
    model_final.save('../Model/best_fake_news_model.keras')
    print("Model disimpan ke '../Model/best_fake_news_model.keras'")


Model terbaik yang akan disimpan adalah: XGBoost dengan fitur FastText
Menyimpan model XGBoost...
Model disimpan ke '../Model/best_fake_news_model.joblib'
